# 09 Thermal network + extra atomic BOP (IA-THM/AIR/FLD/HEX)

**Requirement:** `IA-THM-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

$$
\newcommand{\per}{/}
\newcommand{\squared}{^{2}}
\newcommand{\cubed}{^{3}}
\newcommand{\of}[1]{\,\mathrm{#1}}
\newcommand{\milli}{\mathrm{m}}
\newcommand{\centi}{\mathrm{c}}
\newcommand{\micro}{\mathrm{\mu}}
\newcommand{\kilo}{\mathrm{k}}
\newcommand{\metre}{\mathrm{m}}
\newcommand{\gram}{\mathrm{g}}
\newcommand{\kilogram}{\mathrm{kg}}
\newcommand{\second}{\mathrm{s}}
\newcommand{\minute}{\mathrm{min}}
\newcommand{\hour}{\mathrm{h}}
\newcommand{\day}{\mathrm{d}}
\newcommand{\ampere}{\mathrm{A}}
\newcommand{\volt}{\mathrm{V}}
\newcommand{\watt}{\mathrm{W}}
\newcommand{\ohm}{\mathrm{\Omega}}
\newcommand{\kelvin}{\mathrm{K}}
\newcommand{\celsius}{{}^{\circ}\mathrm{C}}
\newcommand{\coulomb}{\mathrm{C}}
\newcommand{\joule}{\mathrm{J}}
\newcommand{\pascal}{\mathrm{Pa}}
\newcommand{\litre}{\mathrm{L}}
\newcommand{\mole}{\mathrm{mol}}
\newcommand{\molar}{\mathrm{M}}
\newcommand{\percent}{\%}
\newcommand{\atm}{\mathrm{atm}}
\newcommand{\newton}{\mathrm{N}}
\newcommand{\qty}[2]{#1\,#2}
\newcommand{\qtyrange}[3]{#1\text{--}#2\,#3}
\newcommand{\unit}[1]{#1}
\newcommand{\si}[1]{#1}
$$


## Governing equations

Lumped energy balance on electrode, electrolyte, vessel wall, lid, and cavity coolant
(US FIGS. 1E-1F, 9A), with $T$ in $\unit{\kelvin}$ and heat rates in $\unit{\watt}$:

$$m c_p \frac{\mathrm{d}T}{\mathrm{d}t} = \sum q$$

Extra atomic pieces live here as importable modules:
`AirSystem`, `Fan`, `ElectrolyteFlow`, `Pump`, `Reservoir`, `Valve`, `Pipe`, `HeatExchanger`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path

setup_path()
from plant_sim.params import CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G, default_params
from plant_sim.siunitx_nb import SIUNITX_PREAMBLE

In [ ]:
from IPython import get_ipython
from IPython.display import Markdown, display

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 18,
        "figure.figsize": (11, 8.5),
        "figure.dpi": 300,
        "figure.facecolor": "w",
        "figure.edgecolor": "k",
    }
)
display(Markdown(SIUNITX_PREAMBLE))
P = default_params()
print("KOH default", P.c_KOH_mol_m3 / 1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G, 2), round(CAPACITY_MAGNETITE_MAH_G, 2))

## Isolated component simulation

In [ ]:
from plant_sim.components.thermal import ThermalNetwork
from plant_sim.components.air_system import AirSystem, Fan
from plant_sim.components.hydraulics import Pump, Valve
from plant_sim.components.base import integrate_component

th = ThermalNetwork(P)
t, y = integrate_component(
    lambda t, y: th.rhs(
        t, y, {"q_reaction_W": 20.0, "q_joule_W": 5.0, "T_amb_K": P.T_ref_K, "mdot_coolant_kg_s": 0.04}, {}
    ),
    th.y0(),
    (0, 1200),
    n_eval=120,
)
plot_frame = pd.DataFrame(
    {
        "t": (t / 60),
        "Te": (y[0]),
        "Tely": (y[1]),
        "Tv": (y[2]),
        "Tlid": (y[3]),
        "Tc": (y[4]),
    }
)
plot_long = plot_frame.melt(id_vars="t", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="t", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("t (min)")
ax.set_ylabel("T (K)")
ax.set_title("Thermal nodes")
ax.text(0.0, -0.22, "IA-THM-001", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)
for label, obj, uu in [
    ("air", AirSystem(P), {"mdot_air_kg_s": 0.002, "r_orr_mol_s": 1e-6, "T_K": P.T_ep_sim_K}),
    ("fan", Fan(P), {"I_fan_A": 1.0}),
    ("pump", Pump(P), {"I_pump_A": 1.5}),
    ("valve", Valve(P), {"valve_cmd": 1.0, "dP_Pa": 2e4}),
]:
    tt, yy = integrate_component(lambda t, y, o=obj, u=uu: o.rhs(t, y, u, {}), obj.y0(), (0, 15), n_eval=30)
    print(label, "finite", np.all(np.isfinite(yy)))

## Verification against patents / SSS002

In [ ]:
print("Adiabatic node with +q raises T (IA-THM-001 comment). Extra atomic modules imported above.")